## DATOS en Bases de datos SQL

#### La mayoría de las empresas guardan gran cantidad de datos en BBDD relaciones. Aquí van unas nociones para poder "manejar" estos datos cuando tengamos esas entradas en proyectos de Big Data. En clusters de Cloudera, es común encontrarse tablas SQL en HIVE o IMPALA

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# Conectar a la base de datos (se crea si no existe)
conn = sqlite3.connect("test.db")
cursor = conn.cursor()

In [ ]:
# Crear tabla
cursor.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    edad INTEGER,
    email TEXT UNIQUE
)
""")
conn.commit()

In [ ]:
# Insertar datos de ejemplo
usuarios = [
    ("Alice", 25, "alice@example.com"),
    ("Bob", 30, "bob@example.com"),
    ("Charlie", 22, "charlie@example.com")
]

In [ ]:
cursor.executemany("INSERT INTO usuarios (nombre, edad, email) VALUES (?, ?, ?)", usuarios)
conn.commit()

In [ ]:
# Leer los datos
cursor.execute("SELECT * FROM usuarios")
datos = cursor.fetchall()

In [ ]:
# Convertir a DataFrame
df = pd.DataFrame(datos, columns=["id", "nombre", "edad", "email"])
df.head()

,id,nombre,edad,email
0,1,Alice,25,alice@example.com
1,2,Bob,30,bob@example.com
2,3,Charlie,22,charlie@example.com


In [ ]:
query = 'SELECT * FROM usuarios'

In [ ]:
df_pandas = pd.read_sql_query(query, conn, index_col = 'id')
df_pandas.head()

,nombre,edad,email
id,,,
1,Alice,25,alice@example.com
2,Bob,30,bob@example.com
3,Charlie,22,charlie@example.com


In [ ]:
# Cerrar la conexión
conn.close()

#### EL OBJETIVO ES LLEGAR A CONVERTIR LA FUENTE EN UN DF DE PANDAS

## DATOS EN XML

In [ ]:
# Método 1
df1 = pd.read_xml("prueba.xml")

In [ ]:
df1.head()

,name,email,grade,age
0,John,john@mail.com,A,16
1,Alice,alice@mail.com,B,17
2,Bob,bob@mail.com,C,16
3,Hannah,hannah@mail.com,A,17


In [ ]:
## Cargamos el fichero xml

import xml.etree.ElementTree as et

xtree = et.parse("prueba.xml")
xroot = xtree.getroot()

#### EL OBJETIVO ES LLEGAR A CONVERTIR LA FUENTE EN UN DF DE PANDAS

In [ ]:
## Lo guardamos en un DataFrame

df_cols = ["name", "email", "grade", "age"]
rows = []

for node in xroot:
    s_name = node.attrib.get("name")
    s_mail = node.find("email").text
    s_grade = node.find("grade").text
    s_age = node.find("age").text
    rows.append({"name": s_name, "email": s_mail,
                 "grade": s_grade, "age": s_age})

out_df = pd.DataFrame(rows, columns = df_cols)

In [ ]:
out_df

,name,email,grade,age
0,John,john@mail.com,A,16
1,Alice,alice@mail.com,B,17
2,Bob,bob@mail.com,C,16
3,Hannah,hannah@mail.com,A,17


### web scrapping tablas

In [ ]:
import requests

url2 = "http://www.nationmaster.com/country-info/stats/Media/Internet-users"
res = requests.get(url2)

In [ ]:
df = pd.read_html(res.content)

In [ ]:
df[0]

,#,COUNTRY,AMOUNT,DATE,GRAPH,HISTORY
0,1,China,389 million,2009,NaN,NaN
1,2,United States,245 million,2009,NaN,NaN
2,3,Japan,99.18 million,2009,NaN,NaN
3,NaN,Group of 7 countries (G7) average (profile),80.32 million,2009,NaN,NaN
4,4,Brazil,75.98 million,2009,NaN,NaN
...,...,...,...,...,...,...
244,214,Niue,1100,2009,NaN,NaN
245,=215,"Saint Helena, Ascension, and Tristan da Cunha",900,2009,NaN,NaN
246,=215,Saint Helena,900,2009,NaN,NaN
247,217,Tokelau,800,2008,NaN,NaN


#### scraping html WIKIPEDIA

### web scrapping complejos: vamos a sacar los encabezados h2

In [ ]:
from bs4 import BeautifulSoup

url = "https://es.wikipedia.org/wiki/Web_scraping"
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
}

r = requests.get(url, headers=headers, timeout=30)

soup = BeautifulSoup(r.content, "html.parser")

print('tablas')
print(pd.read_html(r.content))

print('resto de elementos')
title = soup.title.get_text(strip=True) if soup.title else None
print("title:", title)

h2s = soup.find_all("h2")

[h.get_text(" ", strip=True) for h in h2s[:10]]

tablas
[                        0                                                  1
0  Control de autoridades  Proyectos Wikimedia  Datos: Q665452  Multimedi...]
resto de elementos
title: Web scraping - Wikipedia, la enciclopedia libre


['Contenidos',
 'Técnicas',
 'Cuestiones legales',
 'Medidas para detener a los scrapers',
 'Beneficios',
 'Herramientas notables',
 'Véase también',
 'Referencias']

## DATOS en API's (JSON)

In [ ]:
import requests

people = requests.get('http://api.open-notify.org/astros.json').json()
print(people)

{'people': [{'craft': 'ISS', 'name': 'Oleg Kononenko'}, {'craft': 'ISS', 'name': 'Nikolai Chub'}, {'craft': 'ISS', 'name': 'Tracy Caldwell Dyson'}, {'craft': 'ISS', 'name': 'Matthew Dominick'}, {'craft': 'ISS', 'name': 'Michael Barratt'}, {'craft': 'ISS', 'name': 'Jeanette Epps'}, {'craft': 'ISS', 'name': 'Alexander Grebenkin'}, {'craft': 'ISS', 'name': 'Butch Wilmore'}, {'craft': 'ISS', 'name': 'Sunita Williams'}, {'craft': 'Tiangong', 'name': 'Li Guangsu'}, {'craft': 'Tiangong', 'name': 'Li Cong'}, {'craft': 'Tiangong', 'name': 'Ye Guangfu'}], 'number': 12, 'message': 'success'}


## DATOS en HDF5

In [ ]:
import numpy as np
import h5py as h5
import pandas as pd

l1 = np.random.random(size = (10000,20))
l2 = np.random.random(size = (10000,200))
df_hdf5 = h5.File('prueba.h5', 'w')
df_hdf5.create_dataset('df1', data=l1)
df_hdf5.create_dataset('df2', data=l2)
df_hdf5.close()

In [ ]:
df_hdf5 =  h5.File('prueba.h5', 'r')
for key in df_hdf5.keys():
    print(key)
df1 = df_hdf5.get('df1')

df1
df2
